# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data

A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model

## DAY 4: Neural Networks and LLMs

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!


In [1]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


/Users/tayjiasheng/AI Projects/llm_engineering/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ["HF_TOKEN"]
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(
    f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items"
)

README.md:   0%|          | 0.00/735 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


# Before we look at the Artificial Neural Networks

## There is a different kind of Neural Network we could consider (a human!)


In [4]:
# Write the test set to a CSV

with open("human_in.csv", "w", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [5]:
# Read it back in

human_predictions = []
with open("human_out.csv", "r", encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [ ]:
def human_pricer(item):
    idx = test.index(item)  # looking up that specific item
    return human_predictions[idx]  # retrieving the result

In [7]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


Human predicted 120.0 for an item that actually costs 219.0


In [8]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

# And now - a vanilla Neural Network

During the remainder of this course we will get deeper into how Neural Networks work, and how to train a neural network.

This is just a sneak preview - let's make our own Neural Network, from scratch, using Pytorch.

Use this to get intuition; it's not important to know all about Neural networks at this point..


In [ ]:
# Prepare our documents and prices

y = np.array(
    [float(item.price) for item in train]
)  # take the price of each item in the training dataset and make it into a numpy array
documents = [item.summary for item in train]

In [10]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors" (vectors that contains 0s and 1s, depending on whether it is present or not present)

# using CountVectorizer to make a vector for each of our documents. A vector that is just composed of numbers (HashingVectorizer is more efficient than CountVectorizer, however it cannot tell you which words is selected, everything is turned into hashes and we cannot see which english words are picked - we use CountVectorizer previously to see which words are picked)
np.random.seed(42)
vectorizer = HashingVectorizer(
    n_features=5000, stop_words="english", binary=True
)  # binary means 0 means not there, 1 means there (can be there like 100 times, 500 times, it will still show 1.)
X = vectorizer.fit_transform(documents)

In [ ]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network (Pytorch makes it easy to create neural networks)

# There's no definition as to what makes a neural network, a deep neural network (which is just a neural network with many layers)

# but 8 layers is still like a just a neural network, not deep neural network yet.
# a good thing about this neural network is that you are not engineering anything, you will let it work out which parameters are good as it undergoes training
class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    # a forward pass
    def forward(self, x):
        output1 = self.relu(
            self.layer1(x)
        )  # each layer is put through a relu activation
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [12]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train_tensor, y_train_tensor, test_size=0.01, random_state=42
)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(
    f"Number of trainable parameters: {trainable_params:,}"
)  # compared to traditional ML, 600_000 parameters is HUGE.

Number of trainable parameters: 669,249


In [ ]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)  # lr is the learning rate.

# We will do 2 complete runs through the data

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize

        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}"
    )

  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 4887.505, Val Loss: 21351.779


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 14999.253, Val Loss: 18101.670


In [ ]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform(
            [item.summary]
        )  # turn it into a vector, 5000 numbers of 0s & 1s
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [16]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$62 $85 $11 $27 $46 $170 $4 $64 $38 $107 $449 $102 $94 $184 $26 $36 $0 $30 $36 $25 $22 $42 $79 $66 $274 $249 $263 $32 $49 $33 $53 $174 $6 $29 $132 $259 $62 $139 $95 $72 $150 $99 $21 $87 $90 $51 $98 $68 $1 $15 $17 $42 $51 $22 $116 $84 $41 $155 $58 $37 $66 $12 $36 $8 $441 $131 $5 $262 $8 $282 $11 $38 $90 $127 $7 $66 $45 $50 $31 $71 $60 $150 $30 $27 $17 $49 $31 $131 $178 $134 $27 $88 $30 $16 $37 $40 $66 $79 $118 $211 $23 $55 $6 $20 $0 $78 $100 $266 $12 $91 $7 $70 $160 $42 $4 $179 $158 $57 $73 $44 $27 $248 $38 $35 $59 $17 $25 $185 $82 $34 $54 $124 $143 $26 $89 $28 $85 $79 $24 $93 $53 $150 $3 $171 $256 $86 $81 $268 $35 $24 $24 $187 $18 $58 $32 $112 $183 $11 $39 $5 $89 $17 $3 $30 $453 $20 $32 $5 $17 $43 $19 $23 $222 $63 $56 $15 $24 $0 $21 $133 $344 $23 $100 $14 $56 $117 $29 $45 $22 $17 $71 $64 $78 $53 $3 $36 $76 $44 $7 $25 

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

Tomorrow we will do some training.

Note: Frontier models are transformers, which is the somewhat like the vanilla neural networks, has many layers, EXCEPT it has attention - it can figure out which part ABOVE in the previous layers MATTERS. It got something to do with embeddings, vector embeddings. (its a fair bit more sophisticated, with billions MORE parameters.)


In [17]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [18]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [19]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [ ]:
# The function for gpt-4.1-nano


def gpt_4__1_nano(item):
    response = completion(
        model="openai/gpt-4.1-nano", messages=messages_for(item)
    )  # you can use openai.chat.completions.create; why 4.1 -> use a cheap model to start with for fine-tuning later one.
    return response.choices[0].message.content

In [21]:
gpt_4__1_nano(test[0])

'$220'

In [22]:
test[0].price

219.0

In [23]:
evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$31 $34 $25 $20 $120 $30 $6 $65 $10 $870 $263 $20 $30 $9 $19 $8 $71 $5 $140 $39 $64 $26 $65 $25 $182 $204 $305 $5 $251 $64 $30 $29 $10 $60 $35 $119 $60 $26 $6 $18 $155 $55 $20 $105 $70 $0 $27 $13 $65 $52 $23 $114 $275 $0 $147 $14 $8 $50 $48 $4 $86 $8 $11 $40 $179 $15 $90 $295 $25 $74 $17 $8 $130 $4 $15 $21 $176 $5 $8 $3 $30 $3 $5 $64 $16 $10 $32 $56 $0 $6 $13 $20 $5 $20 $4 $108 $9 $7 $20 $325 $20 $3 $11 $19 $101 $82 $10 $380 $6 $49 $10 $86 $89 $68 $54 $80 $5 $5 $64 $47 $24 $311 $80 $16 $0 $10 $5 $81 $29 $89 $79 $13 $65 $5 $85 $0 $85 $10 $78 $62 $56 $149 $45 $8 $124 $118 $15 $440 $15 $8 $6 $244 $2 $10 $1 $129 $101 $41 $30 $25 $211 $17 $7 $0 $140 $3 $752 $25 $5 $5 $5 $3 $120 $8 $32 $101 $3 $57 $29 $23 $246 $15 $150 $1 $30 $8 $63 $7 $20 $12 $15 $49 $5 $11 $15 $70 $30 $20 $21 $1 

In [27]:
def claude_opus_4_5(item):
    response = completion(
        model="openrouter/anthropic/claude-opus-4-5", messages=messages_for(item)
    )
    return response.choices[0].message.content

In [ ]:
# evaluate(claude_opus_4_5, test)

In [ ]:
def gemini_3_pro_preview(item):
    response = completion(
        model="gemini/gemini-3-pro-preview",
        messages=messages_for(item),
        reasoning_effort="low",
    )
    return response.choices[0].message.content

In [ ]:
evaluate(
    gemini_3_pro_preview, test, size=50, workers=2
)  # size is the number of datapoints, workers is the number of threads for parallel processing.

In [ ]:
def gemini_2__5_flash_lite(item):
    response = completion(
        model="gemini/gemini-2.5-flash-lite", messages=messages_for(item)
    )
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_2__5_flash_lite, test)

In [ ]:
def grok_4__1_fast(item):
    response = completion(
        model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42
    )
    return response.choices[0].message.content

In [ ]:
evaluate(grok_4__1_fast, test)

In [ ]:
# The function for gpt-5.1


def gpt_5__1(item):
    response = completion(
        model="gpt-5.1", messages=messages_for(item), reasoning_effort="high", seed=42
    )
    return response.choices[0].message.content


In [ ]:
evaluate(gpt_5__1, test)

In [32]:
# The function for gpt-5.1


def gpt_oss_120b(item):
    response = completion(
        model="openrouter/openai/gpt-oss-120b",
        messages=messages_for(item),
        allowed_openai_params=[
            "reasoning_effort"
        ],  # need to include this for openrouter.
        reasoning_effort="low",
        seed=42,
    )
    return response.choices[0].message.content


In [33]:
evaluate(gpt_oss_120b, test, size=50)


  0%|          | 0/50 [00:00<?, ?it/s]

$40 $4 $21 $21 $9 $110 $104 $70 $14 $95 $524 $100 $30 $16 $44 $11 $20 $2 $60 $1 $85 $26 $14 $75 $212 $284 $154 $1 $41 $60 $75 $21 $19 $37 $84 $80 $70 $21 $45 $29 $155 $40 $17 $186 $140 $1 $7 $18 $66 $103 